# LeakageLab: Temporal ML Benchmark
Can a model predict adverse events on future transactions when labels arrive seven days late?

This experiment constructs causal features, purges temporal partitions, selects
on validation, freezes the decision and grades the future test. Data is synthetic:
50,000 transactions, seed 1803; CPU only.

Reusable implementation and tests live in `src/` and `tests/`. On Kaggle, attach
the private **LeakageLab Source** input. Locally, open this notebook from the repo.
An independent portfolio project developed with AI assistance.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import sys
import json
import subprocess
import time
import platform
import shutil

import pandas as pd
from IPython.display import Image, display

ON_KAGGLE = Path('/kaggle/working').exists()
INPUT_ROOT = Path('/kaggle/input') if ON_KAGGLE else Path('artifacts/local/data/solver').resolve()
WORK_ROOT = Path('/kaggle/working' if ON_KAGGLE else 'artifacts/notebook-local').resolve()
PROJECT = WORK_ROOT / 'leakagelab-source'
OUTPUT = WORK_ROOT / 'leakagelab-run'
os.environ.update(OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
STARTED = time.perf_counter()

In [ ]:
if ON_KAGGLE:
    archives = list(INPUT_ROOT.rglob('leakagelab-source.zip'))
    if archives:
        assert len(archives) == 1, 'Attach exactly one LeakageLab Source input'
        PROJECT.mkdir(parents=True, exist_ok=True)
        with ZipFile(archives[0]) as archive:
            archive.extractall(PROJECT)
    else:
        configs = list(INPUT_ROOT.rglob('configs/config.yaml'))
        assert len(configs) == 1, 'Attach the private LeakageLab Source input'
        shutil.copytree(configs[0].parent.parent, PROJECT, dirs_exist_ok=True)
else:
    REPO = Path.cwd() if Path('src/leakagelab').exists() else Path.cwd().parent
    for folder in ['src', 'tests', 'configs']:
        shutil.copytree(REPO / folder, PROJECT / folder, dirs_exist_ok=True)
    shutil.copy2(REPO / 'pyproject.toml', PROJECT / 'pyproject.toml')

sys.path.insert(0, str(PROJECT / 'src'))
from leakagelab.common import read_config, write_json, sha256
from leakagelab.prepare import prepare
from leakagelab.grade import grade
from leakagelab.report import plots

config = read_config(PROJECT / 'configs/config.yaml')
print(f'{platform.system()} | Python {platform.python_version()} | CPU | seed {config["seed"]}')

## 1. Data and temporal partitions
Prediction time is transaction time. History uses strictly earlier transactions;
target aggregates use only matured training labels. Seven-day purges remove rows
whose labels are unavailable at the training and test cutoffs.

Generated data and outputs go to the working directory. The attached source is
read-only. Author labels stay outside the solver's four permitted CSV inputs.

In [ ]:
manifest = prepare(config, OUTPUT / 'data')
BUNDLE = OUTPUT / 'data/solver'
write_json(OUTPUT / 'results/split_manifest.json', manifest)

partitions = pd.DataFrame(manifest['partitions']).T
display(partitions[['n', 'positives', 'prevalence', 'start', 'end']])
print(f'Rows purged: {manifest["purged_rows"]:,}')
assert 'target' not in pd.read_csv(BUNDLE / 'test.csv').columns

## Reference solver (separate process)
Four fixed candidates: prior dummy, logistic regression, and two bounded HistGradientBoosting settings. History ablation and random-split diagnostics are excluded from selection. The random split is INVALID as a future-deployment estimate and uses no final-test labels.

In [ ]:
env = dict(os.environ, PYTHONPATH=str(PROJECT / 'src'))
completed = subprocess.run([sys.executable, '-m', 'leakagelab.solver', '--bundle', str(BUNDLE),
    '--output', str(OUTPUT / 'results'), '--seed', str(config['seed']), '--threads', '2'],
    cwd=BUNDLE, env=env, capture_output=True, text=True, timeout=1800)
print(completed.stdout)
print(completed.stderr)
completed.check_returncode()
LOCK_HASH = sha256(OUTPUT / 'results/selection_lock.json')
print('Frozen selection lock:', LOCK_HASH)


## Author grader: final temporal test
AP is the non-interpolated sklearn average precision, higher is better. Threshold is frozen at validation maximum F1. Entity subgroups and account-cluster bootstrap are secondary, conditional on one synthetic seed; shared time/merchant shocks are not covered.

In [ ]:
result = grade(OUTPUT / 'data/author/test_labels.csv', OUTPUT / 'results/submission.csv',
    OUTPUT / 'results/selection_lock.json', BUNDLE / 'test.csv', BUNDLE / 'train.csv', 200)
assert sha256(OUTPUT / 'results/selection_lock.json') == LOCK_HASH
write_json(OUTPUT / 'results/metrics.json', result)
display(pd.Series(result['test'], name='Future test').to_frame())
display(pd.DataFrame(result['secondary']).T)
print('AP interval:', result['uncertainty']['ap_95_interval'])
plots(OUTPUT)
display(pd.read_csv(OUTPUT / 'results/experiments.csv'))
for name in ['validation_ap.png', 'test_precision_recall.png', 'weekly_prevalence.png']:
    display(Image(filename=str(OUTPUT / 'results' / name)))


## Verification from clean state
Reproducible hashes, strict time/label boundaries, future-invariance, train-only preprocessing, solver input audit, invalid submissions, shuffled-ID joins, and sklearn metric parity.

In [ ]:
checks = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT, env=env,
    capture_output=True, text=True, timeout=1800)
print(checks.stdout)
print(checks.stderr)
(OUTPUT / 'results/tests.log').write_text(checks.stdout + checks.stderr, encoding='utf-8')
checks.check_returncode()
source_files = [p for pattern in ['src/leakagelab/*.py', 'tests/*.py', 'configs/*.yaml', 'pyproject.toml']
                for p in sorted(PROJECT.glob(pattern))]
write_json(OUTPUT / 'results/verification.json', {
    'status': 'success', 'platform': 'Kaggle' if ON_KAGGLE else 'local notebook',
    'cpu_only': True, 'tests_exit_code': checks.returncode,
    'wall_seconds': time.perf_counter() - STARTED,
    'source_hashes': {p.relative_to(PROJECT).as_posix(): sha256(p) for p in source_files},
    'selection_lock_sha256': LOCK_HASH,
    'submission_sha256': sha256(OUTPUT / 'results/submission.csv'),
    'generator_hash': manifest['generator_hash'], 'features_hash': manifest['features_hash']})
# Only the solver-visible bundle is eligible for later sharing; no author labels.
shutil.make_archive(str(WORK_ROOT / 'leakagelab-solver-bundle'), 'zip', BUNDLE)
shutil.make_archive(str(WORK_ROOT / 'leakagelab-results'), 'zip', OUTPUT / 'results')
print('LEAKAGELAB_COMPLETE', json.dumps(result['test']))


## Limitations and reproducibility
The public generator can reproduce labels; withholding CSV ground truth is not anti-cheating security. One synthetic seed does not establish generalization to real adverse events. Drift doubles the approximate prevalence, so AP across partitions is not directly comparable. Account repetition is intentional. This notebook is an author demonstration; only the separate reference-solver notebook and sanitized bundle define participant inputs. Source is AI-assisted; no Eris runtime compatibility or acceptance is claimed.